In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models
datapath = "../../../desktop/quant/hist/aaplIntra.csv"

In [2]:
df = pd.read_csv(datapath).copy()

In [3]:
df

,Dates,Open,Close,High,Low,Volume,Number Ticks
0,7/1/25 9:30,206.665,206.915,207.08,206.600,1035492,1433
1,7/1/25 9:30,206.910,206.710,206.92,206.500,119487,721
2,7/1/25 9:30,206.730,206.810,206.95,206.695,95679,603
3,7/1/25 9:30,206.840,207.200,207.22,206.790,164543,948
4,7/1/25 9:30,207.200,207.115,207.24,206.980,123276,626
...,...,...,...,...,...,...,...
281104,##########,273.900,273.670,274.60,273.470,95766657,2970
281105,##########,273.670,273.670,273.67,273.670,0,1
281106,12/19/25 15:59,273.690,273.900,273.91,273.600,556667,1933
281107,12/19/25 15:59,273.900,273.670,274.60,273.470,95766657,2970


In [18]:
# 1-step ahead Close price target
df["y"] = df["Close"].shift(-1)

# simple lag features (percentage changes for better scaling)
for k in [1, 2, 3, 5]:
    df[f"ret_lag_{k}"] = df["Close"].pct_change(k)

# add current close as a feature
df["close_norm"] = df["Close"] / df["Close"].rolling(window=20).mean()

# drop last row and any NAs from lags
df = df.dropna()

feature_cols = [c for c in df.columns if c.startswith("ret_lag_") or c == "close_norm"]
X = df[feature_cols].values.astype("float32")
y = df["y"].values.astype("float32")

In [ ]:
# train validation split (random 80/20)
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
n_features = X_train.shape[1]

model = models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(
        1,
        activation=None,  # linear
        kernel_regularizer=regularizers.l2(1e-4)  # ridge penalty
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    verbose=1
)

Epoch 1/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 302us/step - loss: 55805.7188 - val_loss: 68805.8359
Epoch 2/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 285us/step - loss: 49414.7969 - val_loss: 61694.7812
Epoch 3/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 283us/step - loss: 43438.9453 - val_loss: 54976.3984
Epoch 4/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 284us/step - loss: 37844.7344 - val_loss: 48649.6367
Epoch 5/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 284us/step - loss: 32669.3750 - val_loss: 42714.0430
Epoch 6/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 287us/step - loss: 27866.8340 - val_loss: 37170.7031
Epoch 7/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 284us/step - loss: 23509.0332 - val_loss: 32019.0469
Epoch 8/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 299us/step - loss: 19474.3672 - val_loss: 27257.1855
Epoch 9/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 286us/step - loss: 15827.2471 - val_loss: 22885.1133
Epoch 10/100
7027/7027 ━━━━━━━━━━━━━━━━━━━━ 2s 284us/step - loss: 12612.5869 - val_loss: 18906.1328
Epoch 11/

In [23]:
model.save('models/regularizedLinear.keras')